<a href="https://colab.research.google.com/github/Bavesh-08/fake-news-classifier-using-Bi-directional-LSTM/blob/main/fake_news_predction_using_Bidirectional_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df=pd.read_csv('/content/FakeNewsNet.csv')
df.head()

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [ ]:
###Drop Nan Values
df=df.dropna()

In [ ]:
## Get the Independent Features

X=df.drop('real',axis=1)

## Get the Dependent features
y=df['real']

In [ ]:
y.value_counts()

,count
real,
1,17441
0,5755


In [ ]:
X.shape,y.shape

((23196, 4), (23196,))

In [ ]:
import tensorflow as tf

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout

In [ ]:
### Vocabulary size
voc_size=10000

In [ ]:
# inehot encoding

messages=X.copy()
messages['title'][1]

"People's Choice Awards 2018: The best red carpet looks"

In [ ]:
messages.reset_index(inplace=True)


In [ ]:
import nltk
import re
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    print(i)
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

Streaming output truncated to the last 5000 lines.
18196
18197
18198
18199
18200
18201
18202
18203
18204
18205
18206
18207
18208
18209
18210
18211
18212
18213
18214
18215
18216
18217
18218
18219
18220
18221
18222
18223
18224
18225
18226
18227
18228
18229
18230
18231
18232
18233
18234
18235
18236
18237
18238
18239
18240
18241
18242
18243
18244
18245
18246
18247
18248
18249
18250
18251
18252
18253
18254
18255
18256
18257
18258
18259
18260
18261
18262
18263
18264
18265
18266
18267
18268
18269
18270
18271
18272
18273
18274
18275
18276
18277
18278
18279
18280
18281
18282
18283
18284
18285
18286
18287
18288
18289
18290
18291
18292
18293
18294
18295
18296
18297
18298
18299
18300
18301
18302
18303
18304
18305
18306
18307
18308
18309
18310
18311
18312
18313
18314
18315
18316
18317
18318
18319
18320
18321
18322
18323
18324
18325
18326
18327
18328
18329
18330
18331
18332
18333
18334
18335
18336
18337
18338
18339
18340
18341
18342
18343
18344
18345
18346
18347
18348
18349
18350
18351
18352
18353
1

In [ ]:
corpus

['kandi burruss explod rape accus real housew atlanta reunion video',
 'peopl choic award best red carpet look',
 'sophia bush send sweet birthday messag one tree hill co star hilari burton breyton eva',
 'colombian singer maluma spark rumour inappropri relationship aunt',
 'gossip girl year later upper east sider shock world chang pop cultur forev',
 'gwen stefani got dump blake shelton jealousi drama exclus',
 'broward counti sheriff fire lie parkland',
 'amber rose shut french montana date rumor call rapper bruvaaa',
 'mindi kale make first post babi appear disneyland wrinkl time co star',
 'katharin mcphee butcher toni nomin drink',
 'wag miami star ashley nicol robert philip wheeler marri',
 'mel gibson hollywood pedophil nowher left hide',
 'medium tyler henri address chill messag kristin cavallari deceas brother express read',
 'dwt season result week disney night',
 'via cq subscrib tax foundat',
 'reason tarek el moussa overcom latest back injuri',
 'david cassidi cut estrang 

In [ ]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[9076, 3375, 1036, 6383, 9148, 3118, 631, 9768, 92, 6653],
 [5988, 6497, 9046, 8831, 8300, 6872, 2193],
 [6248,
  2601,
  7449,
  6947,
  798,
  6605,
  4491,
  7778,
  9436,
  1497,
  2394,
  7050,
  2158,
  9824,
  7877],
 [220, 4881, 538, 7672, 5157, 1932, 3601, 2],
 [4877, 757, 8079, 4372, 5580, 3855, 587, 7607, 7968, 3644, 3136, 3252, 2662],
 [6229, 517, 4870, 4393, 3359, 3056, 2554, 7539, 1261],
 [4218, 1234, 2713, 6421, 359, 1260],
 [868, 8560, 6080, 9931, 981, 6166, 8727, 5141, 2147, 2752],
 [6435, 9923, 6635, 2334, 7328, 6106, 6217, 2950, 2505, 7653, 1497, 2394],
 [5895, 2007, 5040, 9777, 8505, 5446],
 [3416, 5535, 2394, 6790, 6690, 4355, 2450, 2856, 9061],
 [7261, 279, 9379, 5470, 9988, 3932, 3434],
 [7301, 3154, 1216, 974, 6799, 6605, 3177, 128, 8871, 1808, 7916, 2191],
 [2865, 8545, 2082, 4116, 8018, 3763],
 [358, 857, 3239, 9510, 6604],
 [6804, 8472, 4262, 4409, 3931, 7723, 2213, 6536],
 [9186, 4697, 1382, 8424, 5040, 9697, 8833, 5148, 1830, 5761],
 [6132, 8524, 4402, 373

In [ ]:
# embedding part

sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)
embedded_docs[0]

[[   0    0    0 ... 9768   92 6653]
 [   0    0    0 ... 8300 6872 2193]
 [   0    0    0 ... 2158 9824 7877]
 ...
 [   0    0    0 ... 3786  836 4096]
 [   0    0    0 ... 1372 1626 1261]
 [   0    0    0 ... 1947 1275 9046]]


array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 9076,
       3375, 1036, 6383, 9148, 3118,  631, 9768,   92, 6653], dtype=int32)

In [ ]:
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
## Creating model
embedding_vector_features=40
model1=Sequential()
model1.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model1.add(Bidirectional(LSTM(100)))
model1.add(Dropout(0.3))
model1.add(Dense(1,activation='sigmoid'))
model1.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model1.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
len(embedded_docs),y.shape

(23196, (23196,))

In [ ]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [ ]:
X_final.shape,y_final.shape

((23196, 20), (23196,))

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [ ]:
### Finally Training
model1.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 22s 88ms/step - accuracy: 0.8589 - loss: 0.3325 - val_accuracy: 0.8265 - val_loss: 0.4071
Epoch 2/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.8829 - loss: 0.2828 - val_accuracy: 0.8192 - val_loss: 0.4152
Epoch 3/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 16s 66ms/step - accuracy: 0.9001 - loss: 0.2458 - val_accuracy: 0.8063 - val_loss: 0.4433
Epoch 4/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - accuracy: 0.9214 - loss: 0.2047 - val_accuracy: 0.8060 - val_loss: 0.4999
Epoch 5/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 17s 70ms/step - accuracy: 0.9358 - loss: 0.1725 - val_accuracy: 0.7901 - val_loss: 0.5415
Epoch 6/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 19s 65ms/step - accuracy: 0.9469 - loss: 0.1460 - val_accuracy: 0.7974 - val_loss: 0.6068
Epoch 7/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 16s 66ms/step - accuracy: 0.9552 - loss: 0.1257 - val_accuracy: 0.7858 - val_loss: 0.7109
Epoch 8/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 17s 72ms/step - accuracy: 0.9610 - loss: 0.1117 - 

In [ ]:
y_pred1=(model1.predict(X_test) > 0.5).astype("int32")

240/240 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step


In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(y_test,y_pred1)

array([[ 935,  919],
       [ 595, 5206]])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred1)

0.8022207707380797

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred1))

              precision    recall  f1-score   support

           0       0.61      0.50      0.55      1854
           1       0.85      0.90      0.87      5801

    accuracy                           0.80      7655
   macro avg       0.73      0.70      0.71      7655
weighted avg       0.79      0.80      0.80      7655

